# ДЗ9 : Градиентный бустинг (собственная реализация)
## Задача регрессии

In [1]:
import pandas as pd
import numpy as np

from sklearn.datasets import fetch_california_housing, load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, f1_score, accuracy_score, recall_score, precision_score
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import GridSearchCV

import xgboost as xgb
import lightgbm as lgb
import catboost as cb

In [2]:
class MyBoostRegressor:
    def __init__(self, n=400, learning_rate=0.05, depth=7, seed=42, subsample=1.0, colsample_bytree=1.0) -> None:
        self.n = n
        self.learning_rate = learning_rate
        self.depth = depth
        self.seed = seed
        self.trees = []
        self.subsample = subsample
        self.colsample_bytree = colsample_bytree
        self.features_indices = []
        self.feature_importances = []

    def fit(self, X, y):
        if hasattr(X, 'values'):
            X = X.values
        if hasattr(y, 'values'):
            y = y.values
                    
        X = np.asarray(X)
        y = np.asarray(y)
        rng = np.random.default_rng(self.seed)
        self.initial_leaf = y.mean()
        predictions = np.zeros_like(y) + self.initial_leaf
        n_samples, n_features = X.shape
        total_importances = np.zeros(n_features)



        for _ in range(self.n):
            antigrad = y - predictions

            if self.subsample < 1.0:
                n_rows = int(len(X) * self.subsample)
                row_indices = rng.choice(n_samples, size=n_rows, replace=False)
            else:
                row_indices = np.arange(n_samples)

            if self.colsample_bytree < 1.0:
                n_cols = int(n_features * self.colsample_bytree)
                col_indices = rng.choice(n_features, size=n_cols, replace=False)
            else:
                col_indices = np.arange(n_features)

            self.features_indices.append(col_indices)
            X_train = X[row_indices][:, col_indices]
            antigrad_train = antigrad[row_indices]
            tree = DecisionTreeRegressor(max_depth=self.depth, random_state=self.seed, criterion="friedman_mse")
            tree.fit(X_train, antigrad_train)
            self.trees.append(tree)
            total_importances[col_indices] += tree.feature_importances_
            predictions += tree.predict(X[:, col_indices]) * self.learning_rate

        self.feature_importances = total_importances / self.n

    def predict(self, X):
        if hasattr(X, 'values'):
            X = X.values
        X = np.asarray(X)

        predictions = np.zeros(len(X)) + self.initial_leaf
        for i in range(self.n):
            col_indices = self.features_indices[i]
            predictions += self.learning_rate * self.trees[i].predict(X[:, col_indices])
        return predictions

## Задача классификации

In [3]:
class MyBoostClassifier:
    def __init__(self, n=400, learning_rate=0.05, depth=7, seed=42) -> None:
        self.n = n
        self.learning_rate = learning_rate
        self.depth = depth
        self.seed = seed
        self.trees = []
        self.n_classes = None
        self.classes_ = None

    def fit(self, X, y):
        self.classes_ = np.unique(y)
        self.n_classes = len(self.classes_)
        
        self.trees = [[] for _ in range(self.n_classes)]
        
        y_one_hot = np.zeros((len(y), self.n_classes))
        for i, class_label in enumerate(self.classes_):
            y_one_hot[:, i] = (y == class_label).astype(int)

        initial_log_odds = []
        for i in range(self.n_classes):
            p = y_one_hot[:, i].mean()
            p = np.clip(p, 1e-15, 1 - 1e-15)
            initial_log_odds.append(np.log(p / (1 - p)))
        
        self.initial_log_odds = np.array(initial_log_odds)
        predictions = np.full((len(y), self.n_classes), self.initial_log_odds)

        for _ in range(self.n):
            for class_idx in range(self.n_classes):
                exp_pred = np.exp(predictions)
                softmax = exp_pred / exp_pred.sum(axis=1, keepdims=True)
                
                gradient = y_one_hot[:, class_idx] - softmax[:, class_idx]
                
                tree = DecisionTreeRegressor(max_depth=self.depth, random_state=self.seed,criterion="friedman_mse")
                tree.fit(X, gradient)
                self.trees[class_idx].append(tree)
                
                predictions[:, class_idx] += self.learning_rate * tree.predict(X)

    def predict_proba(self, X):
        predictions = np.full((len(X), self.n_classes), self.initial_log_odds)

        for class_idx in range(self.n_classes):
            for tree in self.trees[class_idx]:
                predictions[:, class_idx] += self.learning_rate * tree.predict(X)

        exp_pred = np.exp(predictions - np.max(predictions, axis=1, keepdims=True))
        proba = exp_pred / exp_pred.sum(axis=1, keepdims=True)
        
        return proba
    
    def predict(self, X):
        proba = self.predict_proba(X)
        return self.classes_[np.argmax(proba, axis=1)]

## Сравнение моделей xgBoost, lightGbm, catBoost и MyBoostRegressor на датасете калифорнийских домов

In [4]:
data_reg = fetch_california_housing()
X_reg = pd.DataFrame(data_reg.data, columns=data_reg.feature_names)
y_reg = data_reg.target

X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

models = {
    'XGBoost': {
        'model': xgb.XGBRegressor(n_estimators=400, random_state=42, verbosity=0),
        'params': {
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth': [5, 7, 10],
            'subsample': [0.8, 1]
        }
    },
    'LightGBM': {
        'model': lgb.LGBMRegressor(n_estimators=400, random_state=42, verbose=-1),
        'params': {
            'learning_rate': [0.01, 0.05, 0.1],
            'num_leaves': [31, 70, 127],
            'subsample': [0.8, 1]
        }
    },
    'CatBoost': {
        'model': cb.CatBoostRegressor(iterations=400, random_seed=42, verbose=0),
        'params': {
            'learning_rate': [0.01, 0.05, 0.1],
            'depth': [5, 7, 10],
            'subsample': [0.8, 1]
        }
    }
}

def my_grid_search(models_dict, X_train, y_train, X_test, y_test, cv=5):
    results = {}
    
    for name, config in models_dict.items():
        grid_search = GridSearchCV(
            estimator=config['model'],
            param_grid=config['params'],
            cv=cv,
            scoring='neg_mean_squared_error',
            n_jobs=-1,
            verbose=1
        )
        
        grid_search.fit(X_train, y_train)
        
        results[name] = {
            'best_params': grid_search.best_params_,
            'best_score': -grid_search.best_score_,
            'test_score': mean_squared_error(y_test, grid_search.predict(X_test)),
            'model': grid_search.best_estimator_
        }
        
        print(f"Best params: {grid_search.best_params_}")
        print(f"Best CV score (RMSE): {np.sqrt(-grid_search.best_score_):.4f}")
    
    return results

grid_reg_results = my_grid_search(models, X_reg_train, y_reg_train, X_reg_test, y_reg_test)


Fitting 5 folds for each of 18 candidates, totalling 90 fits
Best params: {'learning_rate': 0.05, 'max_depth': 7, 'subsample': 0.8}
Best CV score (RMSE): 0.4540
Fitting 5 folds for each of 18 candidates, totalling 90 fits
Best params: {'learning_rate': 0.05, 'num_leaves': 70, 'subsample': 0.8}
Best CV score (RMSE): 0.4539
Fitting 5 folds for each of 18 candidates, totalling 90 fits
Best params: {'depth': 10, 'learning_rate': 0.1, 'subsample': 1}
Best CV score (RMSE): 0.4517


In [5]:

results_reg = []
for name in['XGBoost', 'LightGBM', 'CatBoost', 'MyBoostRegressor']:

    if name == 'XGBoost':
        params = grid_reg_results[name]['best_params']
        model = xgb.XGBRegressor(n_estimators=400, random_state=42, n_jobs=-1, **params)
    elif name == 'LightGBM':
        params = grid_reg_results[name]['best_params']
        model = lgb.LGBMRegressor(n_estimators=400, random_state=42, n_jobs=-1, verbose=-1, **params)
    elif name == 'MyBoostRegressor':
        model = MyBoostRegressor()
    else:
        params = grid_reg_results[name]['best_params']
        model = cb.CatBoostRegressor(iterations=400, random_seed=42, thread_count=-1, verbose=0, **params)

    model.fit(X_reg_train, y_reg_train)

    preds_reg = model.predict(X_reg_test)

    mae = mean_absolute_error(y_reg_test, preds_reg)
    rmse = np.sqrt(mean_squared_error(y_reg_test, preds_reg))
    r2 = r2_score(y_reg_test, preds_reg)

    results_reg.append({
        "Модель": name,
        "MAE": round(mae, 4),
        "RMSE": round(rmse, 4),
        "R2 Score": round(r2, 4)
    })

df_results = pd.DataFrame(results_reg)
display(df_results)

,Модель,MAE,RMSE,R2 Score
0,XGBoost,0.2889,0.4445,0.8492
1,LightGBM,0.2848,0.4401,0.8522
2,CatBoost,0.2930,0.4457,0.8484
3,MyBoostRegressor,0.2956,0.4565,0.8410


## Сравнение моделей xgBoost, lightGbm, catBoost и MyBoostClassifier на датасете Iris

In [6]:
data_class = load_iris()
X_class = pd.DataFrame(data_class.data, columns=data_class.feature_names)
y_class = pd.Series(data_class.target)

X_class_train, X_class_test, y_class_train, y_class_test = train_test_split(X_class, y_class, test_size=0.2, random_state=42, stratify=y_class)

model = MyBoostClassifier(n=500, learning_rate=0.01)
model.fit(X_class_train, y_class_train)
pred_class = model.predict(X_class_test)
print("---My Model---")
print(f1_score(y_class_test, pred_class, average='macro'))
print(precision_score(y_class_test, pred_class, average='macro'))
print(accuracy_score(y_class_test, pred_class))

xgb_class = xgb.XGBClassifier(learning_rate=0.01, n_estimators=500, random_state=42, max_depth=7)
xgb_class.fit(X_class_train, y_class_train)
xgb_pred_class = xgb_class.predict(X_class_test)
print("---XGB---")
print(f1_score(y_class_test, xgb_pred_class, average='macro'))
print(precision_score(y_class_test, xgb_pred_class, average='macro'))
print(accuracy_score(y_class_test, xgb_pred_class))

l_class = lgb.LGBMClassifier(learning_rate=0.01, n_estimators=500, random_state=42, num_leaves=31, depth=7)
l_class.fit(X_class_train, y_class_train)
l_pred_class = l_class.predict(X_class_test)
print("---LightGBM---")
print(f1_score(y_class_test, l_pred_class, average='macro'))
print(precision_score(y_class_test, l_pred_class, average='macro'))
print(accuracy_score(y_class_test, l_pred_class))



---My Model---
0.9333333333333332
0.9333333333333332
0.9333333333333333
---XGB---
0.9333333333333332
0.9333333333333332
0.9333333333333333
---LightGBM---
0.899749373433584
0.9023569023569024
0.9
